# Econometrics Lab — Heteroskedasticity and Autocorrelation

**Sorbonne Data Analytics — November 2025**

In this lab we practice the key concepts from the lecture on violations of the classical OLS assumptions. We will work through:

1. **Visualizing the problem** — what heteroskedasticity and autocorrelation look like
2. **Consequences on OLS** — why standard errors become wrong
3. **Testing for heteroskedasticity** — Breusch-Pagan and White tests
4. **Correcting for heteroskedasticity** — Robust (White) standard errors and WLS
5. **Testing for autocorrelation** — Durbin-Watson and Breusch-Godfrey tests
6. **Correcting for autocorrelation** — quasi-differencing (Cochrane-Orcutt)
7. **Newey-West standard errors** — robust to both problems

> **Reminder:** Under heteroskedasticity or autocorrelation, OLS coefficients are still **unbiased**, but their standard errors (and therefore all tests) are **wrong**.


---
## Setup


In [ ]:
# Run this cell first
import numpy as np
import statsmodels.api as sm

np.random.seed(42)
print("All libraries loaded successfully!")

---
## Part 1 — What Does Heteroskedasticity Look Like?

**Homoskedasticity** means $V(u_i) = \sigma^2$ for all $i$ — the variance of the error term is constant.

**Heteroskedasticity** means $V(u_i)$ varies across observations. A classic example: when modeling household spending as a function of income, richer households show more variation in spending.

We will generate two datasets — one homoskedastic, one heteroskedastic — and compare.


### Exercise 1.1 — Generate and visualize both cases

**Dataset A (homoskedastic):** $y_i = 2 + 3 x_i + u_i$ where $u_i \sim N(0, 4)$ for all $i$.

**Dataset B (heteroskedastic):** $y_i = 2 + 3 x_i + u_i$ where $u_i \sim N(0, (0.5 x_i)^2)$ — the variance **grows with** $x$.

Generate $N = 200$ observations with $x \sim U(1, 10)$. For each dataset, make a scatter plot of $y$ vs $x$.

**What do you observe?**


In [ ]:
N = 200
x = np.random.uniform(1, 10, N)

# Dataset A: homoskedastic errors
u_homo = np.random.normal(0, 2, N)
y_homo = 2 + 3 * x + u_homo

# Dataset B: heteroskedastic errors (std dev = 0.5 * x)
u_hetero = np.random.normal(0, 0.5 * x)
y_hetero = 2 + 3 * x + u_hetero

# YOUR CODE HERE — make two scatter plots side by side
# Hint: fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# axes[0].scatter(x, y_homo, ...)
# axes[1].scatter(x, y_hetero, ...)

### Exercise 1.2 — Look at the residuals

Estimate OLS on each dataset and plot the residuals $\hat{u}_i$ against $x_i$.

For the homoskedastic case, the residuals should look like a random cloud with constant spread. For the heteroskedastic case, you should see a **fan shape** (spread increases with $x$).


In [ ]:
# YOUR CODE HERE
# Step 1: Estimate OLS for dataset A
# X = sm.add_constant(x)
# model_homo = sm.OLS(y_homo, X).fit()

# Step 2: Estimate OLS for dataset B
# model_hetero = sm.OLS(y_hetero, X).fit()

# Step 3: Plot residuals vs x for both models (side by side)
# Hint: model_homo.resid gives the residuals



---
## Part 2 — Consequences of Heteroskedasticity on OLS

From the lecture, when $V(u) \neq \sigma^2 I_N$:
- OLS estimates $\hat{b}$ are still **unbiased** — the coefficients are correct on average
- But $\hat{V}(\hat{b}) = \hat{\sigma}^2 (X'X)^{-1}$ is **wrong** — the reported standard errors are incorrect
- Therefore all **t-tests and F-tests are invalid**

Let's verify this with a simulation.


### Exercise 2.1 — OLS is still unbiased under heteroskedasticity

Simulate 1000 datasets from the heteroskedastic model:
$$y_i = 2 + 3 x_i + u_i, \quad u_i \sim N(0, (0.5 x_i)^2)$$

For each simulation, estimate OLS and save $\hat{b}_1$ (coefficient on $x$). Then compute the mean of all 1000 estimates. It should be close to 3 (the true value).


In [ ]:
# YOUR CODE HERE
n_sim = 1000
N = 200
b1_estimates = []

for i in range(n_sim):
    x = np.random.uniform(1, 10, N)
    u = np.random.normal(0, 0.5 * x)
    y = 2 + 3 * x + u

    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    b1_estimates.append(model.params[1])

# Print the mean of b1 estimates
# YOUR CODE: print(f"Mean of b1_hat: {np.mean(b1_estimates):.4f}")
# YOUR CODE: print(f"True b1: 3.0000")

### Exercise 2.2 — But the standard errors are wrong

Using the **heteroskedastic** data generated earlier (dataset B), compare:
1. The standard OLS standard errors (from `model.bse`)
2. The **robust (White)** standard errors

In statsmodels, robust standard errors are obtained with:
```python
model.get_robustcov_results(cov_type='HC1')
```

**Are the standard OLS standard errors too small or too large?**


In [ ]:
# Re-estimate on dataset B
np.random.seed(42)
N = 200
x = np.random.uniform(1, 10, N)
u_hetero = np.random.normal(0, 0.5 * x)
y_hetero = 2 + 3 * x + u_hetero

X = sm.add_constant(x)
model_hetero = sm.OLS(y_hetero, X).fit()

# YOUR CODE HERE
# Step 1: Print standard OLS standard errors
# print("OLS standard errors:", model_hetero.bse)

# Step 2: Compute robust standard errors
# model_robust = model_hetero.get_robustcov_results(cov_type='HC1')
# print("Robust standard errors:", model_robust.bse)

# Step 3: Compare and comment

---
## Part 3 — Testing for Heteroskedasticity

Two classical tests:

**Breusch-Pagan test:** Regress $\hat{u}^2$ on all explanatory variables. Under $H_0$ (homoskedasticity), the test statistic follows a $\chi^2$ distribution.

**White test:** Regress $\hat{u}^2$ on all explanatory variables, their squares, and their cross-products. More general — it detects any form of heteroskedasticity.


### Exercise 3.1 — Breusch-Pagan test (manual)

Perform the Breusch-Pagan test **manually**:

1. Run OLS on the original model, save residuals $\hat{u}$
2. Compute $\hat{u}^2$
3. Regress $\hat{u}^2$ on $x$ (with a constant)
4. The test statistic is $N \cdot R^2$ from this auxiliary regression, which follows $\chi^2_1$ under $H_0$
5. Compute the p-value and conclude


In [ ]:
# Using dataset B (heteroskedastic)
# model_hetero was estimated above

# YOUR CODE HERE
# Step 1: Get residuals
# resid = model_hetero.resid

# Step 2: Compute squared residuals
# resid_sq = resid ** 2

# Step 3: Auxiliary regression: resid_sq = a + b*x
# X_aux = sm.add_constant(x)
# model_aux = sm.OLS(resid_sq, X_aux).fit()

# Step 4: Test statistic = N * R^2
# bp_stat = N * model_aux.rsquared

# Step 5: P-value from chi-squared with 1 df
# p_value = 1 - stats.chi2.cdf(bp_stat, df=1)

# Print results and conclude



### Exercise 3.2 — Breusch-Pagan test (using statsmodels)

Verify your result using `statsmodels.stats.diagnostic.het_breuschpagan`.


In [ ]:
# YOUR CODE HERE
# Hint: bp_result = het_breuschpagan(model_hetero.resid, model_hetero.model.exog)
# bp_result returns (LM stat, LM p-value, F stat, F p-value)
# print(f"LM statistic: {bp_result[0]:.4f}")
# print(f"P-value: {bp_result[1]:.4f}")



### Exercise 3.3 — White test

Run the White test using `het_white`. Compare its conclusion with the Breusch-Pagan test.


In [ ]:
# YOUR CODE HERE
# Hint: white_result = het_white(model_hetero.resid, model_hetero.model.exog)
# white_result returns (LM stat, LM p-value, F stat, F p-value)



### Exercise 3.4 — Test on the homoskedastic dataset

Now run both tests on **dataset A** (homoskedastic). Do the tests correctly fail to reject $H_0$?


In [ ]:
# YOUR CODE HERE
# Estimate OLS on dataset A and run both tests



---
## Part 4 — Correcting for Heteroskedasticity

Two main approaches:

**Approach 1 — Robust (White) standard errors:** Keep the OLS estimates (they are unbiased) but correct the standard errors. This is the most common approach in practice.

**Approach 2 — Weighted Least Squares (WLS):** Transform the model by dividing each observation by $\sqrt{\hat{V}(u_i)}$ so that the transformed errors are homoskedastic.


### Exercise 4.1 — Robust standard errors

Compare three sets of results on dataset B:
1. OLS with standard (wrong) standard errors
2. OLS with robust (White HC1) standard errors
3. OLS with robust (White HC3) standard errors (better for small samples)

For each, report the coefficient estimates, standard errors, t-statistics and p-values.


In [ ]:
# YOUR CODE HERE
# model_ols = sm.OLS(y_hetero, X).fit()
# model_hc1 = sm.OLS(y_hetero, X).fit(cov_type='HC1')
# model_hc3 = sm.OLS(y_hetero, X).fit(cov_type='HC3')

# Print summaries or compare manually
# Note: the COEFFICIENTS are the same — only the standard errors change!



### Exercise 4.2 — Weighted Least Squares (WLS) step by step

Follow the procedure from the lecture:

1. Estimate OLS on the original model, save residuals
2. Run the auxiliary regression: $\log(\hat{u}^2) = \gamma_0 + \gamma_1 x + v$
3. Compute the predicted values $\hat{h}_i = \exp(\hat{\gamma}_0 + \hat{\gamma}_1 x_i)$ — these estimate $V(u_i)$
4. Divide each observation by $\sqrt{\hat{h}_i}$: transform $y$, $x$, and the constant
5. Run OLS on the transformed data (this is WLS)

Compare WLS results with the robust OLS results from Exercise 4.1.


In [ ]:
# YOUR CODE HERE

# Step 1: OLS residuals (already computed)
# resid = model_hetero.resid

# Step 2: Auxiliary regression on log(resid^2)
# log_resid_sq = np.log(resid ** 2)
# model_aux = sm.OLS(log_resid_sq, sm.add_constant(x)).fit()

# Step 3: Predicted variance for each observation
# h_hat = np.exp(model_aux.fittedvalues)

# Step 4: Compute weights = 1 / sqrt(h_hat)
# weights = 1.0 / np.sqrt(h_hat)

# Step 5: Transform all variables
# y_wls = y_hetero * weights
# x_wls = x * weights
# const_wls = np.ones(N) * weights  # the constant becomes 1/sqrt(h)
# X_wls = np.column_stack([const_wls, x_wls])

# Step 6: OLS on transformed data (= WLS)
# model_wls = sm.OLS(y_wls, X_wls).fit()
# print(model_wls.summary())



### Exercise 4.3 — WLS using statsmodels directly

Statsmodels provides `sm.WLS` which does this automatically. Use it with `weights = 1/h_hat`.

Compare with your manual WLS result.


In [ ]:
# YOUR CODE HERE
# model_wls_auto = sm.WLS(y_hetero, sm.add_constant(x), weights=1.0/h_hat).fit()
# print(model_wls_auto.summary())



---
## Part 5 — Autocorrelation: Visualization and Testing

**Autocorrelation** means $\text{Cov}(u_t, u_{t-1}) \neq 0$. This typically occurs in **time series** data.

We model first-order autocorrelation as:
$$u_t = \rho \, u_{t-1} + \varepsilon_t, \quad |\rho| < 1, \quad \varepsilon_t \sim N(0, \sigma_\varepsilon^2)$$

When $\rho > 0$: positive shocks tend to be followed by positive shocks (the most common case in economics).


### Exercise 5.1 — Generate autocorrelated data

Generate a time series dataset with $T = 100$ observations from:
$$y_t = 5 + 2 x_t + u_t, \quad u_t = 0.8 \, u_{t-1} + \varepsilon_t, \quad \varepsilon_t \sim N(0, 1)$$

Note that $\rho = 0.8$ (strong positive autocorrelation).

Then plot the residuals from an OLS regression **over time**. Do you see patterns?


In [ ]:
T = 100
x_ts = np.random.normal(0, 1, T)

# Generate autocorrelated errors: u_t = 0.8 * u_{t-1} + eps_t
rho_true = 0.8
eps = np.random.normal(0, 1, T)
u_ts = np.zeros(T)
u_ts[0] = eps[0]
for t in range(1, T):
    u_ts[t] = rho_true * u_ts[t - 1] + eps[t]

y_ts = 5 + 2 * x_ts + u_ts

# YOUR CODE HERE
# Step 1: Estimate OLS
# X_ts = sm.add_constant(x_ts)
# model_ts = sm.OLS(y_ts, X_ts).fit()

# Step 2: Plot residuals over time (index = time)
# plt.figure(figsize=(10, 4))
# plt.plot(model_ts.resid, '-o', markersize=3)
# plt.axhline(y=0, color='red', linestyle='--')
# plt.xlabel('Time'); plt.ylabel('Residual')
# plt.title('OLS Residuals over Time')
# plt.show()

# What do you observe?

### Exercise 5.2 — Durbin-Watson test

Compute the Durbin-Watson statistic manually:
$$DW = \frac{\sum_{t=2}^{T}(\hat{u}_t - \hat{u}_{t-1})^2}{\sum_{t=1}^{T} \hat{u}_t^2}$$

Recall that $DW \approx 2(1 - \hat{\rho})$. So:
- $DW \approx 2$: no autocorrelation ($\rho \approx 0$)
- $DW \approx 0$: strong positive autocorrelation ($\rho \approx 1$)
- $DW \approx 4$: strong negative autocorrelation ($\rho \approx -1$)

Then verify with `durbin_watson()` from statsmodels.


In [ ]:
# YOUR CODE HERE
# resid_ts = model_ts.resid

# Step 1: Manual DW
# numerator = np.sum(np.diff(resid_ts)**2)
# denominator = np.sum(resid_ts**2)
# dw_manual = numerator / denominator

# Step 2: Implied rho
# rho_hat = 1 - dw_manual / 2

# Step 3: statsmodels DW
# dw_auto = durbin_watson(resid_ts)

# Print all results and compare



### Exercise 5.3 — Breusch-Godfrey test

The Breusch-Godfrey test is more general than Durbin-Watson:
- It works for autocorrelation of any order $p$
- It works even when lagged dependent variables are among the regressors

Use `acorr_breusch_godfrey` from statsmodels with $p = 1$.


In [ ]:
# YOUR CODE HERE
# bg_result = acorr_breusch_godfrey(model_ts, nlags=1)
# bg_result returns (LM stat, LM p-value, F stat, F p-value)
# print(f"LM statistic: {bg_result[0]:.4f}")
# print(f"P-value: {bg_result[1]:.4f}")



### Exercise 5.4 — Test with no autocorrelation

Generate data with **no autocorrelation** ($u_t \sim N(0, 1)$, iid) and verify that the tests correctly fail to reject $H_0$.


In [ ]:
# YOUR CODE HERE
# y_no_ac = 5 + 2 * x_ts + np.random.normal(0, 1, T)
# Run OLS, compute DW, run Breusch-Godfrey



---
## Part 6 — Correcting for Autocorrelation

From the lecture, the correction involves **quasi-differencing**: for $t \geq 2$:
$$y_t - \rho \, y_{t-1} = b_0(1 - \rho) + b_1(x_t - \rho \, x_{t-1}) + \varepsilon_t$$

The first observation is handled separately: multiplied by $\sqrt{1 - \rho^2}$.

Since $\rho$ is unknown, we estimate it from the OLS residuals (two-step or iterative method).


### Exercise 6.1 — Estimate $\rho$ from the residuals

From the OLS model on the autocorrelated data (Exercise 5.1):

1. Compute $\hat{\rho}$ as: $\hat{\rho} = \frac{\sum \hat{u}_t \hat{u}_{t-1}}{\sum \hat{u}_{t-1}^2}$
2. Compare with the true $\rho = 0.8$


In [ ]:
# YOUR CODE HERE
# resid_ts = model_ts.resid
# rho_hat = np.sum(resid_ts[1:] * resid_ts[:-1]) / np.sum(resid_ts[:-1]**2)
# print(f"Estimated rho: {rho_hat:.4f}")
# print(f"True rho: {rho_true}")



### Exercise 6.2 — Quasi-differencing (Cochrane-Orcutt, manual)

Using $\hat{\rho}$, transform the data and re-estimate:

1. For $t \geq 2$: $y^*_t = y_t - \hat{\rho} \, y_{t-1}$ and $x^*_t = x_t - \hat{\rho} \, x_{t-1}$
2. For $t = 1$: $y^*_1 = \sqrt{1 - \hat{\rho}^2} \, y_1$ and $x^*_1 = \sqrt{1 - \hat{\rho}^2} \, x_1$
3. Run OLS on the transformed data (with a transformed constant)

Compare the coefficients and standard errors with the original OLS.


In [ ]:
# YOUR CODE HERE

# Step 1: Quasi-difference y and x (for t >= 2)
# y_star = y_ts[1:] - rho_hat * y_ts[:-1]
# x_star = x_ts[1:] - rho_hat * x_ts[:-1]

# Step 2: First observation
# y1_star = np.sqrt(1 - rho_hat**2) * y_ts[0]
# x1_star = np.sqrt(1 - rho_hat**2) * x_ts[0]

# Step 3: Combine
# y_qd = np.concatenate([[y1_star], y_star])
# x_qd = np.concatenate([[x1_star], x_star])

# Step 4: The constant also needs to be transformed
# const_qd = np.concatenate([[np.sqrt(1 - rho_hat**2)], np.ones(T-1) * (1 - rho_hat)])

# Step 5: OLS on transformed data
# X_qd = np.column_stack([const_qd, x_qd])
# model_co = sm.OLS(y_qd, X_qd).fit()
# print(model_co.summary())



### Exercise 6.3 — Check that autocorrelation is gone

Run the Durbin-Watson and Breusch-Godfrey tests on the residuals of the quasi-differenced model. Is the autocorrelation gone?


In [ ]:
# YOUR CODE HERE
# dw_co = durbin_watson(model_co.resid)
# print(f"DW after correction: {dw_co:.4f}")
# bg_co = acorr_breusch_godfrey(model_co, nlags=1)
# print(f"Breusch-Godfrey p-value after correction: {bg_co[1]:.4f}")



---
## Part 7 — Newey-West Standard Errors

When you have **both** heteroskedasticity **and** autocorrelation (or you are not sure), you can use **Newey-West (HAC)** standard errors. These are robust to both problems.

In statsmodels:
```python
model.fit(cov_type='HAC', cov_kwds={'maxlags': L})
```
where `L` is the maximum lag for autocorrelation (a common rule of thumb is $L = \lfloor T^{1/4} \rfloor$).


### Exercise 7.1 — Compare all standard errors

Using the autocorrelated time series data from Part 5, compare four sets of standard errors:

1. Standard OLS (wrong under both problems)
2. White robust (corrects for heteroskedasticity only)
3. Newey-West HAC (corrects for both)
4. Cochrane-Orcutt corrected model (from Part 6)


In [ ]:
# YOUR CODE HERE
# X_ts = sm.add_constant(x_ts)

# model_ols = sm.OLS(y_ts, X_ts).fit()
# model_white = sm.OLS(y_ts, X_ts).fit(cov_type='HC1')
# model_nw = sm.OLS(y_ts, X_ts).fit(cov_type='HAC', cov_kwds={'maxlags': 4})

# print("Standard errors comparison:")
# print(f"{'Method':<25} {'SE(const)':<12} {'SE(x)':<12}")
# print(f"{'OLS (standard)':<25} {model_ols.bse[0]:<12.4f} {model_ols.bse[1]:<12.4f}")
# print(f"{'White (HC1)':<25} {model_white.bse[0]:<12.4f} {model_white.bse[1]:<12.4f}")
# print(f"{'Newey-West (HAC)':<25} {model_nw.bse[0]:<12.4f} {model_nw.bse[1]:<12.4f}")
# print(f"{'Cochrane-Orcutt':<25} {model_co.bse[0]:<12.4f} {model_co.bse[1]:<12.4f}")



---
## Summary

| Problem | Detection | Correction |
|---|---|---|
| Heteroskedasticity | Residual plots, Breusch-Pagan, White test | Robust (White) SE, or WLS |
| Autocorrelation | Residual time plot, Durbin-Watson, Breusch-Godfrey | Quasi-differencing (Cochrane-Orcutt) |
| Both | — | Newey-West (HAC) SE |

**Key takeaways:**
- Under heteroskedasticity or autocorrelation, OLS coefficients remain unbiased, but standard errors and tests are wrong
- In practice, using **robust standard errors** (White or Newey-West) is the simplest and most common fix
- WLS and Cochrane-Orcutt are more efficient but require correctly specifying the error structure
